# 02 · Campaign — binder module + switch module

**Standard slot:** *design campaign.* **For Project 12 this is the core:** run the **binder module**
(the Project-06 two-paradigm workflow) against your analyte epitope, **and** design/borrow the
**switch module** (an RFdiffusion scaffold or a LOCKR-style cage), then write the results CSVs (D2):
- **Binder — BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **Binder — RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.
- **Switch** — a handful of switch scaffolds (split-reporter and/or LOCKR-style), scored on their
  intrinsic toggle quality.

> **Compute honesty:** a real binder campaign + switch modeling at this scale wants an **A100**
> (Colab Pro+ or a cluster). Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a
> small RFdiffusion batch + ESMFold triage). The cells below run on the deterministic **mock** backend
> so the plumbing executes anywhere; the real calls + A100 notes are shown alongside. Run
> `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The binder and switch tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs
still exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and
log it). The generation itself needs an A100; this check needs nothing.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # binder paradigm #1; pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # binder mode + scaffold/switch; pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer / two-state modeling; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")
print("Switch literature (no install): Langan 2019 LOCKR; Quijano-Rubio 2021 biosensors; Dixon 2016 NanoBiT.")

## 1 · Define the campaign

Same analyte + epitope as notebook 01. Set honest campaign sizes; the cells run on `mock` so they
execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the numbers on
a T4** (FreeBindCraft, a small RFdiffusion batch).

In [ ]:
import biosensor_tools as bt
import pandas as pd

ANALYTE  = "ANALYTE"                          # STUDENT CHOICE — your verified biomarker target
HOTSPOTS = bt.parse_hotspots("A12,A45,A60")   # EXAMPLE — replace with your verified epitope residues
READOUT  = "split_luciferase"                 # split_luciferase | nanobit | split_fluorophore_fret
SWITCH_FAMILY = "split_reporter"              # "split_reporter" or "lockr"

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4
N_SWITCH      = 8       # a handful of switch scaffolds to compare architectures

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab
TOOL_SWITCH      = "mock"   # -> "rfdiffusion" (scaffold) or "lockr" (borrow a cage) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"Switch      : n={N_SWITCH} family={SWITCH_FAMILY} reporter={READOUT} tool={TOOL_SWITCH}")
print("analyte/epitope:", ANALYTE, HOTSPOTS)

## 2 · Binder paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/biosensor_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

## 3 · Binder paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the epitope, then ProteinMPNN designs sequences, then AF2-Multimer
re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate is low — that
is normal). The `mock` backend stands in for the whole chain.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

## 4 · Switch module — design/borrow the transduction element

The switch converts binding into signal. Two routes (see `MANUAL.md §1`):
- **split-reporter:** an RFdiffusion scaffold presenting a split-luciferase/NanoBiT or split-FP FRET
  pair that reconstitutes on binding;
- **LOCKR-style cage:** borrow/adapt a published de novo cage+latch (Langan 2019), graft the
  reporter/functional element, redesign the latch so the analyte displaces it.

We score each switch on its **intrinsic** toggle quality (state separation + OFF-state leak) — this is
analyte-independent; the binder-coupled ON/OFF behaviour comes in notebook 04. Mock numbers are
SYNTHETIC.

In [ ]:
# Real call (Colab, A100): bt.design_switch(scaffold=..., reporter=READOUT, family=SWITCH_FAMILY,
#   n=N_SWITCH, tool="rfdiffusion")  # or tool="lockr" to borrow/adapt a published LOCKR cage.
switches = bt.design_switch(scaffold="rfdiff_scaffold", reporter=READOUT,
                            family=SWITCH_FAMILY, n=N_SWITCH, tool=TOOL_SWITCH)
# Also compare a LOCKR-style family so notebook 04's architecture benchmark has two architectures.
switches_lockr = bt.design_switch(scaffold="lockr_cage", reporter="",
                                  family="lockr", n=N_SWITCH, tool=TOOL_SWITCH)
print(f"switch pool: {len(switches)} {SWITCH_FAMILY} + {len(switches_lockr)} lockr (SYNTHETIC if mock)")
s = switches[0]
print("example switch:", s.switch_id, "toggle_score=", s.toggle_score, "background_leak=", s.background_leak)

## 5 · Assemble + persist the pools

Write one tidy CSV per binder paradigm (plus a combined one) and one for the switches. These feed
notebook 03 (the shared **binder** filter) and notebook 04 (integration + ON/OFF). We add an EXAMPLE
physics column (`rosetta_dG`) here so the binder physics layer has something to act on in the dry run
— on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [ ]:
import pandas as pd

def binder_pool_to_df(designs):
    rows = []
    for d in designs:
        # Mock dry run: attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is exercised.
        # On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            epitope_coverage=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

def switch_pool_to_df(switches):
    return pd.DataFrame([dict(
        switch_id=s.switch_id, family=s.family, reporter=s.reporter, scaffold=s.scaffold,
        closed_plddt=s.closed_plddt, open_plddt=s.open_plddt,
        toggle_score=s.toggle_score, background_leak=s.background_leak, synthetic=s.synthetic,
    ) for s in switches])

df_bc = binder_pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = binder_pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

df_sw = switch_pool_to_df(list(switches) + list(switches_lockr))
df_sw.to_csv("results/switch_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("wrote results/switch_designs.csv      ", df_sw.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

## D2 checklist
- [ ] BindCraft binder pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4).
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100).
- [ ] Every binder scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Switch module: ≥1 architecture designed/borrowed (split-reporter and/or LOCKR), `results/switch_designs.csv`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared binder filter** on the binder pools.